# Landmark Recgnition - GLDv2

## CSYE7105
## Team 8

In [1]:
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

import argparse
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR


In [2]:
"""
Dataset and data loading utilities for landmark image retrieval.
"""

class LandmarkDataset(Dataset):
    """Dataset for loading landmark images with labels."""

    def __init__(self, csv_path, image_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.transform = transform

        # Build label mapping: landmark_id -> contiguous index
        unique_labels = sorted(self.df["landmark_id"].unique())
        self.label_to_idx = {lbl: idx for idx, lbl in enumerate(unique_labels)}
        self.idx_to_label = {idx: lbl for lbl, idx in self.label_to_idx.items()}
        self.num_classes = len(unique_labels)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, row["image_id"]+".jpg")
        image = Image.open(img_path).convert("RGB")
        label = self.label_to_idx[row["landmark_id"]]

        if self.transform:
            image = self.transform(image)

        return image, label, row["image_id"]


def get_train_transform(image_size=224):
    return transforms.Compose([
        transforms.RandomResizedCrop(image_size, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])


def get_val_transform(image_size=224):
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])


def create_data_loaders(data_dir, batch_size=32, num_workers=4, image_size=224):
    """Create train and validation data loaders."""
    image_dir = os.path.join(data_dir, "images")
    train_csv = os.path.join(data_dir, "train_split.csv")
    val_csv = os.path.join(data_dir, "val_split.csv")

    train_dataset = LandmarkDataset(train_csv, image_dir, get_train_transform(image_size))
    val_dataset = LandmarkDataset(val_csv, image_dir, get_val_transform(image_size))

    # Ensure val uses the same label mapping as train
    val_dataset.label_to_idx = train_dataset.label_to_idx
    val_dataset.idx_to_label = train_dataset.idx_to_label
    val_dataset.num_classes = train_dataset.num_classes

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True, drop_last=True,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
    )

    return train_loader, val_loader, train_dataset.num_classes


In [3]:
"""
Landmark retrieval model with ArcFace head for metric learning.
"""

class ArcFaceHead(nn.Module):
    """ArcFace classification head for metric learning.

    Produces angular-margin softmax logits that encourage the model
    to learn discriminative, well-separated embeddings.
    """

    def __init__(self, embedding_dim, num_classes, s=30.0, m=0.3):
        super().__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, embeddings, labels):
        # Normalize weights and embeddings
        w = F.normalize(self.weight, dim=1)
        x = F.normalize(embeddings, dim=1)

        # Cosine similarity
        cosine = F.linear(x, w)
        cosine = cosine.clamp(-1.0 + 1e-7, 1.0 - 1e-7)

        # ArcFace margin
        theta = torch.acos(cosine)
        target_logits = torch.cos(theta + self.m)

        # One-hot encode labels and apply margin only to target class
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.unsqueeze(1), 1.0)
        logits = one_hot * target_logits + (1.0 - one_hot) * cosine

        return logits * self.s


class LandmarkRetrievalModel(nn.Module):
    """ResNet50 backbone with embedding layer and ArcFace head."""

    def __init__(self, num_classes, embedding_dim=512, device=None):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.device = device if device is not None else torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

        # Backbone: ResNet50 pretrained
        backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        backbone_out_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone

        # Embedding head
        self.embedding = nn.Sequential(
            nn.Linear(backbone_out_dim, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
        )

        # ArcFace classification head (used only during training)
        self.arcface = ArcFaceHead(embedding_dim, num_classes)

        self.to(self.device)

        print(f"Initialized model on {self.device} .")

    def extract_embedding(self, x):
        """Extract L2-normalized embedding for retrieval."""
        features = self.backbone(x)
        emb = self.embedding(features)
        return F.normalize(emb, dim=1)

    def forward(self, x, labels=None):
        """
        Forward pass.
        - During training (labels provided): returns ArcFace logits.
        - During inference (no labels): returns normalized embeddings.
        """
        emb = self.extract_embedding(x)
        if labels is not None:
            return self.arcface(emb, labels)
        return emb

    def train_one_epoch(self, loader, criterion, optimizer):
        """Run one training epoch. Returns (avg_loss, accuracy)."""
        self.train()
        total_loss = 0.0
        correct = 0
        total = 0

        for images, labels, _ in loader:
            images, labels = images.to(self.device), labels.to(self.device)

            logits = self(images, labels)
            loss = criterion(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += images.size(0)

        return total_loss / total, correct / total

    @torch.no_grad()
    def validate(self, loader, criterion):
        """Run validation. Returns (avg_loss, accuracy)."""
        self.eval()
        total_loss = 0.0
        correct = 0
        total = 0

        for images, labels, _ in loader:
            images, labels = images.to(self.device), labels.to(self.device)

            logits = self(images, labels)
            loss = criterion(logits, labels)

            total_loss += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += images.size(0)

        return total_loss / total, correct / total

    def fit(
        self,
        train_loader,
        optimizer,
        scheduler,
        criterion,
        epochs,
        save_dir,
        val_loader=None,
        patience=5,
        plot=False,
    ):
        """
        Full training loop with optional validation, early stopping, and history plotting.

        Args:
            train_loader: DataLoader for training data.
            optimizer: Optimizer instance.
            scheduler: LR scheduler instance.
            criterion: Loss function.
            epochs (int): Maximum number of epochs.
            save_dir (str): Directory to save checkpoints.
            val_loader: DataLoader for validation (enables early stopping). Optional.
            patience (int): Early-stopping patience in epochs (only used when val_loader given).
            plot (bool): If True, plot loss/accuracy curves after training.

        Returns:
            pd.DataFrame: History with columns epoch, train_loss, train_acc,
                          and val_loss, val_acc when val_loader is provided.
        """
        os.makedirs(save_dir, exist_ok=True)

        history = []
        best_val_acc = 0.0
        patience_counter = 0
        last_epoch = epochs
        num_classes = self.arcface.weight.size(0)

        for epoch in range(1, epochs + 1):
            t0 = time.time()

            train_loss, train_acc = self.train_one_epoch(train_loader, criterion, optimizer)
            row = {"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc}

            val_loss, val_acc = None, None
            if val_loader is not None:
                val_loss, val_acc = self.validate(val_loader, criterion)
                row["val_loss"] = val_loss
                row["val_acc"] = val_acc

            scheduler.step()
            history.append(row)

            elapsed = time.time() - t0
            log = (
                f"Epoch {epoch}/{epochs} ({elapsed:.0f}s) | "
                f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f}"
            )
            if val_loader is not None:
                log += f" | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}"
            print(log)

            # Early stopping + best-model checkpoint
            if val_loader is not None:
                if val_acc > best_val_acc:
                    best_val_acc = val_acc
                    patience_counter = 0
                    torch.save(
                        {
                            "epoch": epoch,
                            "model_state_dict": self.state_dict(),
                            "num_classes": num_classes,
                            "embedding_dim": self.embedding_dim,
                            "val_acc": val_acc,
                        },
                        os.path.join(save_dir, "best_model.pth"),
                    )
                    print(f"  -> Saved best model (val_acc={val_acc:.4f})")
                else:
                    patience_counter += 1
                    if patience_counter >= patience:
                        print(
                            f"Early stopping after epoch {epoch} "
                            f"(no improvement for {patience} epochs)."
                        )
                        last_epoch = epoch
                        break

        # Save final model
        torch.save(
            {
                "epoch": last_epoch,
                "model_state_dict": self.state_dict(),
                "num_classes": num_classes,
                "embedding_dim": self.embedding_dim,
                "val_acc": best_val_acc if val_loader is not None else None,
            },
            os.path.join(save_dir, "final_model.pth"),
        )

        history_df = pd.DataFrame(history)

        summary = f"Training complete. Best val accuracy: {best_val_acc:.4f}" if val_loader is not None \
            else "Training complete."
        print(summary)

        if plot:
            self._plot_history(history_df)

        return history_df

    def _plot_history(self, history_df):
        """Plot training (and validation) loss and accuracy curves."""
        try:
            import matplotlib.pyplot as plt
        except ImportError:
            print("matplotlib not installed — skipping plot.")
            return

        has_val = "val_loss" in history_df.columns
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        axes[0].plot(history_df["epoch"], history_df["train_loss"], label="Train Loss")
        if has_val:
            axes[0].plot(history_df["epoch"], history_df["val_loss"], label="Val Loss")
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("Loss")
        axes[0].set_title("Loss History")
        axes[0].legend()
        axes[0].grid(True)

        axes[1].plot(history_df["epoch"], history_df["train_acc"], label="Train Acc")
        if has_val:
            axes[1].plot(history_df["epoch"], history_df["val_acc"], label="Val Acc")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Accuracy")
        axes[1].set_title("Accuracy History")
        axes[1].legend()
        axes[1].grid(True)

        plt.tight_layout()
        plt.show()


In [ ]:
"""
Training script for the landmark retrieval model.

"""

data_dir = "data/my_data"
do_plot = True

save_dir = "checkpoints"

image_size = 224
num_workers = 4

epochs = 10
batch_size = 32
lr = 1e-4
embedding_dim = 512

patience = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data
print("Loading datasets...")
train_loader, val_loader, num_classes = create_data_loaders(
    data_dir, batch_size, num_workers, image_size
)
print(f"Classes: {num_classes}, Train: {len(train_loader.dataset)}, Val: {len(val_loader.dataset)}")

# Model — device is passed in and managed internally
print("Initializing model...")
model = LandmarkRetrievalModel(num_classes, embedding_dim, device=device)

# Optimizer: lower lr for pretrained backbone, higher for new layers
backbone_params = list(model.backbone.parameters())
new_params = list(model.embedding.parameters()) + list(model.arcface.parameters())
optimizer = AdamW([
    {"params": backbone_params, "lr": lr * 0.1},
    {"params": new_params, "lr": lr},
], weight_decay=1e-4)

scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
criterion = nn.CrossEntropyLoss()

print("Training ...")
# Train — returns history DataFrame
history = model.fit(
    train_loader=train_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    epochs=epochs,
    save_dir=save_dir,
    val_loader=val_loader,
    patience=patience,
    plot=do_plot,
)

history_csv = os.path.join(save_dir, "train_history.csv")
history.to_csv(history_csv, index=False)
print(f"History saved to {history_csv}")




Loading datasets...
Classes: 7448, Train: 96502, Val: 9190
Initializing model...
Initialized model on cuda .
Training ...
Epoch 1/10 (372s) | Train Loss: 13.2879 Acc: 0.0403 | Val Loss: 9.0762 Acc: 0.1249
  -> Saved best model (val_acc=0.1249)
